In [12]:
import tifffile as tiff
from pathlib import Path
import pickle
import numpy as np
import pandas as pd

SCRATCH_DIR = Path("/root/capsule/scratch")
MORPH_DIR = SCRATCH_DIR / "morphology"
MORPH_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path("/root/capsule/data")

In [13]:
data_info = pickle.load(open('/root/capsule/code/Preprocessing/hcr_data_info.pkl', 'rb'))
mouse_ids = list(data_info.keys())
print('mouse_ids:', mouse_ids)

mouse_ids: [755252, 767018, 767022, 782149, 788406, 790322]


In [14]:
# np is already imported in an earlier cell; avoid re-importing it here

zstack_size_um = [450, 400, 400]
for mouse_id in mouse_ids:
    output_path = MORPH_DIR / f"{mouse_id}h_mask_properties.csv"
    if output_path.exists():    
        print(f"  → {output_path.name} already exists – skipping.")
        continue
    masks_dir = DATA_DIR / data_info[mouse_id]['cortical_zstack']['segmented'] / "channel_0_ref_0/segmentation_masks.tif"
    masks = tiff.imread(masks_dir)
    
    zstack_size_pixels = masks.shape
    zstack_resolution = [zstack_size_um[i] / zstack_size_pixels[i] for i in range(3)]

    labels = np.unique(masks)
    labels = labels[labels != 0]

    res = np.array(zstack_resolution)  # [z_res, y_res, x_res]
    results = []

    for lab in labels:
        coords = np.column_stack(np.where(masks == lab))  # z,y,x (pixels)
        coords_um = coords * res  # convert to microns
        centroid = coords_um.mean(axis=0)  # z,y,x

        # size in original x,y,z (micron)
        size_z = np.ptp(coords_um[:, 0])
        size_y = np.ptp(coords_um[:, 1])
        size_x = np.ptp(coords_um[:, 2])

        # PCA to get principal axes
        coords_centered = coords_um - centroid
        if coords_centered.shape[0] >= 3:
            _, _, Vt = np.linalg.svd(coords_centered, full_matrices=False)
            pcs = Vt  # rows: principal axes in (z,y,x) coordinates
        else:
            pcs = np.eye(3)

        # sizes along principal axes (micron)
        proj = coords_centered @ pcs.T
        size_pc1 = np.ptp(proj[:, 0])
        size_pc2 = np.ptp(proj[:, 1])
        size_pc3 = np.ptp(proj[:, 2])

        # angle of dominant axis in XY plane (degrees)
        vx = pcs[0, 2]  # x component of first principal axis
        vy = pcs[0, 1]  # y component
        angle_deg_xy = float(np.degrees(np.arctan2(vy, vx)))

        results.append({
            "label": int(lab),
            "centroid_x_um": float(centroid[2]),
            "centroid_y_um": float(centroid[1]),
            "centroid_z_um": float(centroid[0]),
            "angle_deg_xy": angle_deg_xy,
            "size_x_um": float(size_x),
            "size_y_um": float(size_y),
            "size_z_um": float(size_z),
            "size_pc1_um": float(size_pc1),
            "size_pc2_um": float(size_pc2),
            "size_pc3_um": float(size_pc3),
            "n_voxels": int(coords.shape[0])
        })

    df_masks_props = pd.DataFrame(results).sort_values("label").reset_index(drop=True)
    print(df_masks_props)
    df_masks_props.to_csv(output_path, index=False)

     label  centroid_x_um  centroid_y_um  centroid_z_um  angle_deg_xy  \
0        1     322.092223     353.520633      25.841346     31.767092   
1        2     149.742551     234.462624      50.975838     61.507735   
2        3     324.830098     285.505366      52.274036     57.302737   
3        4     336.216222     109.859795      52.107589     62.626430   
4        5      96.112235      46.004891      41.346202    -65.797256   
..     ...            ...            ...            ...           ...   
830    831      24.853215     120.533232     442.599621     43.948303   
831    832      44.336771     289.802632     445.709104     80.918514   
832    833     343.301661     155.960685     445.363982    -63.232712   
833    834      33.672521     175.025181     446.205785     95.536806   
834    835     134.247025     242.269243     446.144082    129.142468   

     size_x_um  size_y_um  size_z_um  size_pc1_um  size_pc2_um  size_pc3_um  \
0     25.78125   18.75000        8.0    26.9

In [15]:
df_masks_props

,label,centroid_x_um,centroid_y_um,centroid_z_um,angle_deg_xy,size_x_um,size_y_um,size_z_um,size_pc1_um,size_pc2_um,size_pc3_um,n_voxels
0,1,393.522335,47.729171,42.938212,-124.962568,13.28125,13.28125,12.0,17.420763,12.214260,12.966485,2282
1,2,295.508989,155.383388,44.969888,-165.084934,14.84375,13.28125,18.0,18.766638,15.260223,13.986868,3487
2,3,394.414688,223.548341,43.706636,-56.792440,9.37500,12.50000,14.0,15.536790,14.809213,10.929909,2185
3,4,374.557732,249.445771,43.911158,-43.806404,14.84375,14.84375,15.0,17.594752,16.987820,15.687752,3782
4,5,270.177606,47.984983,45.299478,-37.872413,17.18750,15.62500,14.0,19.883513,15.725886,13.796601,3446
...,...,...,...,...,...,...,...,...,...,...,...,...
1011,1012,202.471619,45.003743,445.708583,-163.708559,11.71875,8.59375,8.0,14.062875,11.268719,8.969654,1002
1012,1013,258.975087,221.845498,446.661538,-17.202895,13.28125,11.71875,8.0,13.757367,11.291191,8.572256,715
1013,1014,252.515453,28.334478,445.243956,-179.014623,10.93750,10.15625,5.0,10.995103,10.181382,5.031679,455
1014,1015,320.249418,218.944099,447.077640,-6.955003,13.28125,9.37500,6.0,13.314445,9.857310,6.147701,644
